# MobileNetV3 para PlantVillage

Implementacion reproducible de MobileNetV3 como candidato principal para mejorar el baseline MobileNetV2 de la app Android. Mantiene las mismas 38 clases de PlantVillage y registra precision, graficas, tamano TFLite y tiempo de inferencia local.

Decision inicial: usar `MobileNetV3Small` por viabilidad movil. Se puede cambiar a `MobileNetV3Large` si la accuracy no mejora lo suficiente y el tamano/latencia siguen siendo aceptables.

In [2]:
from pathlib import Path
import os
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import tensorflow_datasets as tfds
import keras
from sklearn.metrics import classification_report, confusion_matrix

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_DIR = Path('/Users/carolinachavez/convolutional-neuronal-network/MobileNetV2')
ANDROID_ASSETS_DIR = Path('/Users/carolinachavez/Documents/Leaf-disease-applicaton/Leafdisease/app/src/main/assets')
CONTEXT_DOC = Path('/Users/carolinachavez/Documents/Tesis docs/context/model_mobileNet.md')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
NUM_CLASSES_EXPECTED = 38

MODEL_NAME = 'mobilenetv3_small_v1'
MODEL_VARIANT = 'small'  # 'small' recomendado para movil; 'large' si necesitas mas capacidad.
TRAINING_MODE = 'full'  # 'smoke_test' primero, luego 'full'.

SMOKE_TRAIN_EXAMPLES = 2048
SMOKE_VAL_EXAMPLES = 512
SMOKE_TEST_EXAMPLES = 512

HEAD_EPOCHS = 3 if TRAINING_MODE == 'smoke_test' else 10
FINE_TUNE_EPOCHS = 0 if TRAINING_MODE == 'smoke_test' else 10
HEAD_LR = 1e-3
FINE_TUNE_LR = 1e-5
FINE_TUNE_LAST_N_LAYERS = 30

RUN_TRAINING = True
RUN_FINE_TUNING = TRAINING_MODE == 'full'
RUN_EXPORT = TRAINING_MODE == 'full'
RUN_DATASET_PROFILE = True

print('Python executable:', os.sys.executable)
print('TensorFlow:', tf.__version__)
print('Keras:', keras.__version__)
print('Training mode:', TRAINING_MODE)
print('Model variant:', MODEL_VARIANT)

Python executable: /opt/anaconda3/envs/learning_ai/bin/python
TensorFlow: 2.20.0
Keras: 3.14.0
Training mode: full
Model variant: small


## Carga del dataset y preparacion

Se usa `tensorflow_datasets/plant_village` para mantener compatibilidad con el baseline MobileNetV2. MobileNetV3 se crea con `include_preprocessing=False`, por lo que el notebook normaliza manualmente cada imagen a `[-1, 1]`, alineado con el preprocessing de Android.

In [3]:
def preprocess(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = (image - 127.5) / 127.5
    return image, label

raw_ds, info = tfds.load('plant_village', split='train', as_supervised=True, with_info=True)
class_names = info.features['label'].names
num_classes = info.features['label'].num_classes
total_examples = info.splits['train'].num_examples
assert num_classes == NUM_CLASSES_EXPECTED, f'Expected 38 classes, got {num_classes}'

train_size = int(total_examples * 0.80)
val_size = int(total_examples * 0.10)
test_size = total_examples - train_size - val_size

shuffled = raw_ds.shuffle(buffer_size=min(total_examples, 10_000), seed=SEED, reshuffle_each_iteration=False)
train_raw = shuffled.take(train_size)
remaining = shuffled.skip(train_size)
val_raw = remaining.take(val_size)
test_raw = remaining.skip(val_size)

if TRAINING_MODE == 'smoke_test':
    train_raw = train_raw.take(SMOKE_TRAIN_EXAMPLES)
    val_raw = val_raw.take(SMOKE_VAL_EXAMPLES)
    test_raw = test_raw.take(SMOKE_TEST_EXAMPLES)
    effective_train_size = SMOKE_TRAIN_EXAMPLES
    effective_val_size = SMOKE_VAL_EXAMPLES
    effective_test_size = SMOKE_TEST_EXAMPLES
else:
    effective_train_size = train_size
    effective_val_size = val_size
    effective_test_size = test_size

train_ds = train_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
val_ds = val_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_steps = int(np.ceil(effective_train_size / BATCH_SIZE))
val_steps = int(np.ceil(effective_val_size / BATCH_SIZE))
test_steps = int(np.ceil(effective_test_size / BATCH_SIZE))

print('Total examples:', total_examples)
print('Full split sizes:', {'train': train_size, 'validation': val_size, 'test': test_size})
print('Effective split sizes:', {'train': effective_train_size, 'validation': effective_val_size, 'test': effective_test_size})
print('Estimated steps:', {'train': train_steps, 'validation': val_steps, 'test': test_steps})
print('Number of classes:', num_classes)
print('First labels:', class_names[:5])

Total examples: 54303
Full split sizes: {'train': 43442, 'validation': 5430, 'test': 5431}
Effective split sizes: {'train': 43442, 'validation': 5430, 'test': 5431}
Estimated steps: {'train': 1358, 'validation': 170, 'test': 170}
Number of classes: 38
First labels: ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy']


In [4]:
labels_path = PROJECT_DIR / 'labels_mobilenetv3.txt'
labels_path.write_text('\n'.join(class_names) + '\n')

android_labels_path = ANDROID_ASSETS_DIR / 'labels.txt'
if android_labels_path.exists():
    android_labels = [line.strip() for line in android_labels_path.read_text().splitlines() if line.strip()]
    labels_match_android = android_labels == class_names
else:
    labels_match_android = False

print('Wrote MobileNetV3 labels:', labels_path)
print('Labels match Android asset:', labels_match_android)

Wrote MobileNetV3 labels: /Users/carolinachavez/convolutional-neuronal-network/MobileNetV2/labels_mobilenetv3.txt
Labels match Android asset: True


## Grafica de distribucion de clases

Esta grafica ayuda a detectar desbalance antes del entrenamiento.

In [ ]:
if RUN_DATASET_PROFILE:
    counts = np.zeros(num_classes, dtype=np.int64)
    for _, labels in raw_ds.batch(2048):
        counts += np.bincount(labels.numpy(), minlength=num_classes)

    class_distribution_df = pd.DataFrame({'class_name': class_names, 'count': counts}).sort_values('count', ascending=False)
    plt.figure(figsize=(14, 8))
    sns.barplot(data=class_distribution_df, x='count', y='class_name', color='#4C78A8')
    plt.title('PlantVillage - distribucion de clases')
    plt.xlabel('Cantidad de imagenes')
    plt.ylabel('Clase')
    plt.tight_layout()
    plt.show()
else:
    class_distribution_df = pd.DataFrame()

class_distribution_df.head(10)

## Baseline MobileNetV2 actual

Se registra el tamano del modelo actual integrado en Android para compararlo con la exportacion de MobileNetV3.

In [ ]:
def file_size_mb(path):
    path = Path(path)
    return path.stat().st_size / (1024 * 1024) if path.exists() else None

baseline_tflite = PROJECT_DIR / 'mobilenetv2_v3_fp16.tflite'
android_baseline_tflite = ANDROID_ASSETS_DIR / 'mobilenetv2_v3_fp16.tflite'

pd.DataFrame([{
    'model': 'MobileNetV2 actual FP16',
    'project_size_mb': file_size_mb(baseline_tflite),
    'android_size_mb': file_size_mb(android_baseline_tflite),
    'classes': len(class_names),
}])

## Construccion del modelo MobileNetV3

El backbone se carga con pesos `imagenet`, sin capa superior (`include_top=False`) y congelado al inicio. Encima se agrega una cabeza ligera: global average pooling, dropout y una capa densa de 38 clases con softmax.

In [ ]:
def build_mobilenetv3(num_classes):
    if MODEL_VARIANT == 'large':
        backbone_factory = tf.keras.applications.MobileNetV3Large
    elif MODEL_VARIANT == 'small':
        backbone_factory = tf.keras.applications.MobileNetV3Small
    else:
        raise ValueError("MODEL_VARIANT must be 'small' or 'large'")

    backbone = backbone_factory(
        input_shape=(*IMG_SIZE, 3),
        include_top=False,
        weights='imagenet',
        include_preprocessing=False,
    )
    backbone.trainable = False

    inputs = keras.Input(shape=(*IMG_SIZE, 3), name='image')
    x = backbone(inputs, training=False)
    x = keras.layers.GlobalAveragePooling2D(name='avg_pool')(x)
    x = keras.layers.Dropout(0.2, name='dropout')(x)
    outputs = keras.layers.Dense(num_classes, activation='softmax', name='classifier')(x)

    model = keras.Model(inputs, outputs, name=f'mobilenetv3_{MODEL_VARIANT}_plantvillage')
    return model, backbone

model, backbone = build_mobilenetv3(num_classes)
model.compile(optimizer=keras.optimizers.Adam(learning_rate=HEAD_LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()
print('Backbone params:', backbone.count_params())
print('Trainable params:', sum(np.prod(v.shape) for v in model.trainable_weights))

## Entrenamiento de la cabeza

Primera fase: el backbone queda congelado y solo se entrena la cabeza de clasificacion. Esto adapta las features preentrenadas de ImageNet a las 38 clases de PlantVillage con menor costo.

In [ ]:
checkpoint_path = PROJECT_DIR / f'{MODEL_NAME}_best.keras'
training_log_path = PROJECT_DIR / f'{MODEL_NAME}_training_log.csv'

callbacks = [
    keras.callbacks.ModelCheckpoint(filepath=str(checkpoint_path), monitor='val_accuracy', mode='max', save_best_only=True, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_accuracy', mode='max', patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-7, verbose=1),
    keras.callbacks.CSVLogger(str(training_log_path)),
]

history_head = None
history_finetune = None

if RUN_TRAINING:
    print(f'Training mode: {TRAINING_MODE}. Head epochs: {HEAD_EPOCHS}.')
    print(f'Expected train steps per epoch: {train_steps}; validation steps: {val_steps}.')
    history_head = model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, callbacks=callbacks)
else:
    print('RUN_TRAINING=False: skipping head training.')

## Fine-tuning controlado

Segunda fase opcional: en `TRAINING_MODE = 'full'`, se descongelan solo las ultimas capas del backbone y se entrena con learning rate bajo. BatchNorm queda congelado para estabilidad.

In [ ]:
if RUN_TRAINING and RUN_FINE_TUNING:
    backbone.trainable = True
    for layer in backbone.layers[:-FINE_TUNE_LAST_N_LAYERS]:
        layer.trainable = False
    for layer in backbone.layers:
        if isinstance(layer, keras.layers.BatchNormalization):
            layer.trainable = False

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=FINE_TUNE_LR), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    history_finetune = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=HEAD_EPOCHS + FINE_TUNE_EPOCHS,
        initial_epoch=history_head.epoch[-1] + 1 if history_head else 0,
        callbacks=callbacks,
    )
elif RUN_TRAINING:
    print('RUN_FINE_TUNING=False: skipping fine-tuning for smoke test.')
else:
    print('RUN_TRAINING=False: skipping fine-tuning.')

## Graficas de entrenamiento

Se grafican accuracy y loss para detectar aprendizaje, sobreajuste o estancamiento.

In [ ]:
def history_to_frame(*histories):
    frames = []
    offset = 0
    for name, hist in histories:
        if hist is None:
            continue
        frame = pd.DataFrame(hist.history)
        frame['epoch'] = np.arange(offset, offset + len(frame)) + 1
        frame['phase'] = name
        frames.append(frame)
        offset += len(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

history_df = history_to_frame(('head', history_head), ('fine_tune', history_finetune))

if not history_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.lineplot(data=history_df, x='epoch', y='accuracy', marker='o', label='train', ax=axes[0])
    sns.lineplot(data=history_df, x='epoch', y='val_accuracy', marker='o', label='validation', ax=axes[0])
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')

    sns.lineplot(data=history_df, x='epoch', y='loss', marker='o', label='train', ax=axes[1])
    sns.lineplot(data=history_df, x='epoch', y='val_loss', marker='o', label='validation', ax=axes[1])
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')

    plt.tight_layout()
    plt.show()

history_df

## Evaluacion y matriz de confusion

Se carga el mejor checkpoint si existe, se evalua el test set y se genera reporte por clase.

In [ ]:
if checkpoint_path.exists():
    eval_model = keras.models.load_model(checkpoint_path)
    print('Loaded best checkpoint:', checkpoint_path)
else:
    eval_model = model
    print('Checkpoint not found; using in-memory model.')

test_loss, test_accuracy = eval_model.evaluate(test_ds, verbose=1)
print({'test_loss': test_loss, 'test_accuracy': test_accuracy})

y_true, y_pred, y_conf = [], [], []
for images, labels in test_ds:
    probs = eval_model.predict(images, verbose=0)
    preds = np.argmax(probs, axis=1)
    y_true.extend(labels.numpy().tolist())
    y_pred.extend(preds.tolist())
    y_conf.extend(np.max(probs, axis=1).tolist())

report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()
report_df.head(12)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title('MobileNetV3 - Matriz de confusion PlantVillage')
plt.xlabel('Prediccion')
plt.ylabel('Real')
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
errors = []
for true_idx, pred_idx, conf in zip(y_true, y_pred, y_conf):
    if true_idx != pred_idx:
        errors.append({'true_label': class_names[true_idx], 'predicted_label': class_names[pred_idx], 'confidence': conf})

errors_df = pd.DataFrame(errors)
if not errors_df.empty:
    display(errors_df.sort_values('confidence', ascending=False).head(25))
else:
    print('No errors in this test subset.')

## Exportacion a SavedModel y TFLite

La exportacion queda desactivada en `smoke_test`. Para generar archivos finales, cambiar `TRAINING_MODE = 'full'` y volver a ejecutar el notebook.

In [ ]:
saved_model_dir = PROJECT_DIR / 'saved_models' / MODEL_NAME
fp32_tflite_path = PROJECT_DIR / f'{MODEL_NAME}_fp32.tflite'
fp16_tflite_path = PROJECT_DIR / f'{MODEL_NAME}_fp16.tflite'

export_results = {}
if RUN_EXPORT:
    saved_model_dir.parent.mkdir(parents=True, exist_ok=True)
    eval_model.export(str(saved_model_dir))

    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    tflite_fp32 = converter.convert()
    fp32_tflite_path.write_bytes(tflite_fp32)

    converter = tf.lite.TFLiteConverter.from_saved_model(str(saved_model_dir))
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_types = [tf.float16]
    tflite_fp16 = converter.convert()
    fp16_tflite_path.write_bytes(tflite_fp16)

    export_results = {
        'saved_model': str(saved_model_dir),
        'fp32_tflite': str(fp32_tflite_path),
        'fp32_size_mb': file_size_mb(fp32_tflite_path),
        'fp16_tflite': str(fp16_tflite_path),
        'fp16_size_mb': file_size_mb(fp16_tflite_path),
    }
else:
    export_results['export'] = 'Skipped because TRAINING_MODE is smoke_test.'

export_results

## Benchmark local TFLite y comparacion

Este benchmark no reemplaza una prueba en Android real, pero estima tamano y latencia local antes de integrar el modelo.

In [ ]:
def benchmark_tflite(tflite_path, dataset, warmup=3, runs=25):
    tflite_path = Path(tflite_path)
    if not tflite_path.exists():
        return {'path': str(tflite_path), 'error': 'missing file'}

    interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    sample_images, _ = next(iter(dataset.unbatch().batch(1)))
    sample = sample_images.numpy().astype(input_details['dtype'])

    timings_ms = []
    for i in range(warmup + runs):
        interpreter.set_tensor(input_details['index'], sample)
        start = time.perf_counter()
        interpreter.invoke()
        _ = interpreter.get_tensor(output_details['index'])
        elapsed_ms = (time.perf_counter() - start) * 1000
        if i >= warmup:
            timings_ms.append(elapsed_ms)

    return {'path': str(tflite_path), 'size_mb': file_size_mb(tflite_path), 'avg_ms': float(np.mean(timings_ms)), 'p95_ms': float(np.percentile(timings_ms, 95)), 'runs': runs}

benchmarks = [benchmark_tflite(baseline_tflite, test_ds)]
if fp16_tflite_path.exists():
    benchmarks.append(benchmark_tflite(fp16_tflite_path, test_ds))

benchmark_df = pd.DataFrame(benchmarks)
benchmark_df

In [ ]:
comparison_df = pd.DataFrame([
    {
        'model': 'MobileNetV2 actual FP16',
        'test_accuracy': None,
        'tflite_size_mb': file_size_mb(baseline_tflite),
        'avg_inference_ms_local': next((b.get('avg_ms') for b in benchmarks if b.get('path') == str(baseline_tflite)), None),
        'notes': 'Baseline actual de la app; accuracy no medido aqui.',
    },
    {
        'model': f'MobileNetV3 {MODEL_VARIANT}',
        'test_accuracy': test_accuracy,
        'tflite_size_mb': file_size_mb(fp16_tflite_path),
        'avg_inference_ms_local': next((b.get('avg_ms') for b in benchmarks if b.get('path') == str(fp16_tflite_path)), None),
        'notes': f'Training mode: {TRAINING_MODE}',
    },
])

plt.figure(figsize=(8, 5))
sns.barplot(data=comparison_df.dropna(subset=['tflite_size_mb']), x='model', y='tflite_size_mb', color='#59A14F')
plt.title('Comparacion de tamano TFLite')
plt.xlabel('Modelo')
plt.ylabel('Tamano MB')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

comparison_df

## Registro en contexto

Al finalizar una corrida, esta celda agrega un resumen en `model_mobileNet.md`.

In [ ]:
def append_run_to_context():
    timestamp = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    fp16_size = file_size_mb(fp16_tflite_path)
    lines = [
        '',
        f'## Corrida MobileNetV3 - {timestamp}',
        '',
        f'- Notebook: `{PROJECT_DIR / "MobileNetV3.ipynb"}`',
        f'- Modelo: `MobileNetV3{MODEL_VARIANT.title()}` con pesos ImageNet, `include_preprocessing=False`',
        f'- Training mode: `{TRAINING_MODE}`',
        f'- Dataset: PlantVillage, 38 clases, split 80/10/10, seed `{SEED}`',
        f'- Effective split sizes: train `{effective_train_size}`, validation `{effective_val_size}`, test `{effective_test_size}`',
        f'- Test accuracy: `{test_accuracy:.4f}`',
        f'- Test loss: `{test_loss:.4f}`',
        f'- TFLite FP16 size MB: `{fp16_size}`',
        '- Decision provisional: comparar contra MobileNetV2 por accuracy, tamano y latencia antes de integrar a Android.',
        '',
    ]
    with CONTEXT_DOC.open('a') as f:
        f.write('\n'.join(lines))

append_run_to_context()
print('Updated context:', CONTEXT_DOC)